# 🎬 Ghibli-Style AI Video Prompt Generator

Generate end-to-end, scene-by-scene prompts for creating Studio Ghibli-style AI videos.

This notebook walks you through:
1. **Single prompt generation** — one-off Ghibli-style scene prompts
2. **Full storyboard generation** — multi-scene video with narrative arc
3. **Platform-specific formatting** — optimized for Sora, Runway, Kling, or Pika
4. **Template-only mode** — works without API keys
5. **LLM-powered mode** — uses GPT-4o/Gemini/Claude for intelligent scene expansion

## Setup

Make sure your `.env` file has at least one of the following API keys:
```
OPENAI_API_KEY=sk-...
ANTHROPIC_KEY=sk-ant-...
GEMINI_API_KEY=...
```

Template-only mode works without any API keys.

In [ ]:
from ghibli_prompter import (
    GhibliPromptGenerator,
    generate_single_prompt,
    list_options,
    GHIBLI_ENVIRONMENTS,
    GHIBLI_MOOD_KEYWORDS,
    GHIBLI_LIGHTING,
    CAMERA_MOVEMENTS,
    GHIBLI_COLOR_PALETTES,
)
from llm_agent import LLM, LLMType, Models

## 1. Explore Available Options

See all the Ghibli aesthetic building blocks you can use.

In [ ]:
list_options()

## 2. Generate a Single Ghibli Prompt

Quick way to create one prompt for a specific scene. No LLM needed.

In [ ]:
prompt = generate_single_prompt(
    description="A young witch flies on a broomstick over a coastal town at sunset, her cat perched behind her",
    platform="sora",
    environment="ocean",
    lighting="sunset",
    mood="adventure",
    camera="aerial",
    aspect_ratio="16:9",
    duration=12,
)
print(prompt)

In [ ]:
# Same scene but formatted for Runway
prompt_runway = generate_single_prompt(
    description="A young witch flies on a broomstick over a coastal town at sunset, her cat perched behind her",
    platform="runway",
    environment="ocean",
    lighting="sunset",
    mood="adventure",
    camera="aerial",
    aspect_ratio="16:9",
    duration=12,
)
print(prompt_runway)

## 3. Full Storyboard — Template Mode (No API Keys)

Generate a complete multi-scene storyboard using built-in Ghibli knowledge.
This works entirely offline without any LLM API calls.

In [ ]:
gen = GhibliPromptGenerator()

storyboard = gen.generate_video_prompts(
    concept="A lonely boy discovers a hidden garden where ancient forest spirits dance at twilight",
    num_scenes=6,
    platform="sora",
    aspect_ratio="16:9",
    duration_per_scene=10,
    environment="forest",
    mood="wonder",
    character_description="A quiet boy of ten with messy black hair and an oversized green jacket",
    use_llm=False,  # Template-only mode
)

storyboard.print_storyboard()

In [ ]:
# Extract just the prompts for easy copy-paste into your AI video tool
for p in storyboard.get_all_prompts():
    print(f"\n--- Scene {p['scene']} ({p['type']}) ---")
    print(p['prompt'])

## 4. Full Storyboard — LLM-Powered Mode

Uses GPT-4o (or another LLM) to intelligently expand your concept into
richly detailed scenes with proper narrative arc, then formats each
scene as a platform-ready prompt.

**Requires an API key** in your `.env` file.

In [ ]:
# Option A: Use default (OpenAI GPT-4o)
gen = GhibliPromptGenerator()

# Option B: Use a specific LLM
# gen = GhibliPromptGenerator(llm=LLM(llm_type=LLMType.GEMINI, model=Models.GEMINI_1_5_PRO))
# gen = GhibliPromptGenerator(llm=LLM(llm_type=LLMType.CLAUDE, model=Models.CLAUDE2))

In [ ]:
storyboard_llm = gen.generate_video_prompts(
    concept="A cat spirit who lives in an abandoned train station guides lost travelers to their true destination",
    num_scenes=8,
    platform="sora",
    aspect_ratio="16:9",
    duration_per_scene=10,
    environment="village",
    mood="mystery",
    character_description="A translucent white cat with glowing blue eyes, shimmering softly at the edges",
    use_llm=True,
)

storyboard_llm.print_storyboard()

## 5. Export as JSON

Save the storyboard for programmatic use or to share with collaborators.

In [ ]:
import json

# Print formatted JSON
print(storyboard.to_json())

# Or save to file
# with open("storyboard.json", "w") as f:
#     f.write(storyboard.to_json())

## 6. Platform Comparison

See how the same concept produces different prompts for each platform.

In [ ]:
scene_desc = "A girl sits on a mossy stone bridge, dangling her feet over a crystal-clear stream, watching fish swim below"

for platform in ["sora", "runway", "kling", "pika"]:
    print(f"\n{'='*60}")
    print(f"  PLATFORM: {platform.upper()}")
    print(f"{'='*60}")
    p = generate_single_prompt(
        description=scene_desc,
        platform=platform,
        environment="forest",
        lighting="forest_dappled",
        mood="peace",
        camera="dolly_in",
    )
    print(p)

## 7. Custom Workflow: Build Your Own Storyboard Scene-by-Scene

For maximum control, generate individual scene prompts and assemble them manually.

In [ ]:
from ghibli_prompter import VideoStoryboard, ScenePrompt

# Create an empty storyboard
my_storyboard = VideoStoryboard(
    title="The Wind Carries Seeds",
    concept="A dandelion seed's journey across a Ghibli landscape",
    platform="sora",
    aspect_ratio="16:9",
    total_duration=40,
    style_notes="Macro to wide perspective shifts, following a single seed",
)

# Scene 1: Close-up of a dandelion
p1 = generate_single_prompt(
    description="Extreme close-up of a dandelion head, a single seed detaches as a gentle breeze arrives",
    platform="sora",
    environment="countryside",
    lighting="morning",
    mood="peace",
    camera="close_up",
)
my_storyboard.add_scene(ScenePrompt(
    scene_number=1, scene_type="opening",
    description="A dandelion seed breaks free",
    prompt=p1, camera="Close-up", environment="Meadow",
    lighting="Morning light", mood="Peaceful beginning",
    duration_seconds=10,
))

# Scene 2: The seed drifts over a village
p2 = generate_single_prompt(
    description="A dandelion seed drifts high above a small rural village with red-roofed houses and chimney smoke",
    platform="sora",
    environment="village",
    lighting="midday",
    mood="wonder",
    camera="aerial",
)
my_storyboard.add_scene(ScenePrompt(
    scene_number=2, scene_type="journey",
    description="The seed floats over a village",
    prompt=p2, camera="Aerial", environment="Village",
    lighting="Midday", mood="Wonder",
    duration_seconds=10,
))

# Scene 3: The seed passes through a forest
p3 = generate_single_prompt(
    description="The dandelion seed weaves between ancient tree trunks in a sunlit forest, passing a sleeping fox",
    platform="sora",
    environment="forest",
    lighting="forest_dappled",
    mood="mystery",
    camera="follow",
)
my_storyboard.add_scene(ScenePrompt(
    scene_number=3, scene_type="journey",
    description="Through the ancient forest",
    prompt=p3, camera="Tracking", environment="Forest",
    lighting="Dappled", mood="Mysterious",
    duration_seconds=10,
))

# Scene 4: The seed lands in a garden
p4 = generate_single_prompt(
    description="The dandelion seed gently descends into a sunlit garden where a child looks up in wonder as it lands on their palm",
    platform="sora",
    environment="garden",
    lighting="golden_hour",
    mood="joy",
    camera="dolly_in",
)
my_storyboard.add_scene(ScenePrompt(
    scene_number=4, scene_type="closing",
    description="The seed finds a home",
    prompt=p4, camera="Dolly in", environment="Garden",
    lighting="Golden hour", mood="Joyful resolution",
    duration_seconds=10,
))

my_storyboard.print_storyboard()

---

## Tips for Best Results

**General:**
- Keep one subject/action per scene for clarity
- Use active kinetic verbs (glides, drifts, rushes) not static descriptions
- Maintain the same character description across all scenes for consistency

**Ghibli-Specific:**
- Include "mundane magic" moments (eating, walking, watching nature)
- Describe wind affecting hair, clothing, and foliage
- Add particle effects: floating dust motes, drifting petals, fireflies
- Limit to 3-4 base colors with 1-2 accents per scene
- Use gentle, deliberate camera movements (never jarring cuts)

**Platform-Specific:**
- **Sora**: Best for narrative coherence; specify style blending percentages
- **Runway**: Supports video-to-video; great for converting real footage to Ghibli style
- **Kling**: Include audio cues; has motion brush for fine control
- **Pika**: Best for short loops and quick social media clips